In [1]:
%pip install pandas numpy

  Using cached pandas-3.0.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (79 kB)
Using cached pandas-3.0.1-cp313-cp313-macosx_11_0_arm64.whl (9.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 23.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Load data and print to check

In [2]:
import pandas as pd
import numpy as np 

df: pd.DataFrame = pd.read_csv("../data/raw/cybersecurity_attacks.csv")

#Quick check 
print(f"Dataset loaded with {df.shape[0]} rows")
df.head()

Dataset loaded with 40000 rows


,Timestamp,Source IP Address,Destination IP Address,Source Port,Destination Port,Protocol,Packet Length,Packet Type,Traffic Type,Payload Data,...,Action Taken,Severity Level,User Information,Device Information,Network Segment,Geo-location Data,Proxy Information,Firewall Logs,IDS/IPS Alerts,Log Source
0,2023-05-30 06:33:58,103.216.15.12,84.9.164.252,31225,17616,ICMP,503,Data,HTTP,Qui natus odio asperiores nam. Optio nobis ius...,...,Logged,Low,Reyansh Dugal,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT ...,Segment A,"Jamshedpur, Sikkim",150.9.97.135,Log Data,NaN,Server
1,2020-08-26 07:08:30,78.199.217.198,66.191.137.154,17245,48166,ICMP,1174,Data,HTTP,Aperiam quos modi officiis veritatis rem. Omni...,...,Blocked,Low,Sumer Rana,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT ...,Segment B,"Bilaspur, Nagaland",NaN,Log Data,NaN,Firewall
2,2022-11-13 08:23:25,63.79.210.48,198.219.82.17,16811,53600,UDP,306,Control,HTTP,Perferendis sapiente vitae soluta. Hic delectu...,...,Ignored,Low,Himmat Karpe,Mozilla/5.0 (compatible; MSIE 9.0; Windows NT ...,Segment C,"Bokaro, Rajasthan",114.133.48.179,Log Data,Alert Data,Firewall
3,2023-07-02 10:38:46,163.42.196.10,101.228.192.255,20018,32534,UDP,385,Data,HTTP,Totam maxime beatae expedita explicabo porro l...,...,Blocked,Medium,Fateh Kibe,Mozilla/5.0 (Macintosh; PPC Mac OS X 10_11_5; ...,Segment B,"Jaunpur, Rajasthan",NaN,NaN,Alert Data,Firewall
4,2023-07-16 13:11:07,71.166.185.76,189.243.174.238,6131,26646,TCP,1462,Data,DNS,Odit nesciunt dolorem nisi iste iusto. Animi v...,...,Blocked,Low,Dhanush Chad,Mozilla/5.0 (compatible; MSIE 5.0; Windows NT ...,Segment C,"Anantapur, Tripura",149.6.110.119,NaN,Alert Data,Firewall


Data Cleaning 1,2 and type conversion



In [3]:
#Data cleaning 
#Strategy 1: Drop rows where cirtical info is missing

df.dropna(subset=['Protocol', 'Attack Type'], inplace=True)

#Strategy 2: Fill missing numerical values with Median value to avoid losing any data
median_anomaly: float = df['Anomaly Scores'].median()
df['Anomaly Scores'] = df['Anomaly Scores'].fillna(median_anomaly)

#Type conversion: timestamps are set to real dates
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
print("Data cleaning complete. No more misisng values in critical columns")

Data cleaning complete. No more misisng values in critical columns


Apply functions, export cleaned data


In [4]:
#Applying Functions - custom column "risk level"
df['risk_level'] = np.where(df['Anomaly Scores'] > 85, 'High', 'Low')

#Save to cleaned file
df.to_csv("../data/processed/cleaned_attacks.csv", index=False)
print("Cleaned data exported to data/processed/cleaned_attacks.csv")

Cleaned data exported to data/processed/cleaned_attacks.csv


Data Analysis

In [ ]:
#Data analysis
print(df.describe())


                        Timestamp   Source Port  Destination Port  \
count                       40000  40000.000000      40000.000000   
mean   2021-11-22 06:17:52.200800  32970.356450      33150.868650   
min           2020-01-01 00:43:27   1027.000000       1024.000000   
25%    2020-12-12 19:56:28.500000  16850.750000      17094.750000   
50%           2021-11-21 18:12:51  32856.000000      33004.500000   
75%    2022-10-30 08:48:10.500000  48928.250000      49287.000000   
max           2023-10-11 19:34:23  65530.000000      65535.000000   
std                           NaN  18560.425604      18574.668842   

       Packet Length  Anomaly Scores  
count   40000.000000    40000.000000  
mean      781.452725       50.113473  
min        64.000000        0.000000  
25%       420.000000       25.150000  
50%       782.000000       50.345000  
75%      1143.000000       75.030000  
max      1500.000000      100.000000  
std       416.044192       28.853598  


Do certain network segments carry heavier traffic loads?

In [15]:
#Multi Column Aggregation (Average Packet Length by Segment)
#Do certian network segments carry heavier traffic loads?
segment_stats = df.groupby("Network Segment")["Packet Length"].agg(["mean", "count"])
segment_stats.columns = ["Average Packet Length", "Count"]
segment_stats = segment_stats.sort_values("Average Packet Length", ascending= False)

print(segment_stats)

                 Average Packet Length  Count
Network Segment                              
Segment C                   783.925194  13408
Segment A                   782.823552  13273
Segment B                   777.597642  13319


All three network segments show similar average packet lengths (778–784 bytes), with Segment C carrying the highest average and most traffic overall.

Do higher severity events correspond to higher anomaly scores?

In [16]:
#Average Anomaly Score by Severity
#Do higher severity events correspond to higher anomaly scores?
severity_stats = df.groupby("Severity Level")["Anomaly Scores"].agg(["mean", "count"])
severity_stats.columns = ["Average Anomaly Score", "Count"]
severity_stats = severity_stats.reindex(["Low", "Medium", "High"])

print(severity_stats)

                Average Anomaly Score  Count
Severity Level                              
Low                         50.235696  13183
Medium                      49.808101  13435
High                        50.299650  13382


Anomaly scores are very similar across all severity levels, this suggests that anomaly scores do not strongly differentiate between low, medium, and high severity levels.

Are blocked packets larger than logged or ignored ones?

In [17]:
#Average Packet Length by Action Taken
#Are blocked packets larger than logged or ignored ones?
action_stats = df.groupby("Action Taken")["Packet Length"].agg(["mean", "count"])
action_stats.columns = ["Average Packet Length", "Count"]
action_stats = action_stats.sort_values("Average Packet Length", ascending= False)

print(action_stats)

              Average Packet Length  Count
Action Taken                              
Blocked                  784.173331  13529
Ignored                  781.720398  13276
Logged                   778.393937  13195


Logged packets tend to by a smaller average packet length, while blocked carries the most traffic. This might be because larger packets carry more data which is potentially a risk.

Descriptive Stats and Correlation Analysis

In [ ]:
#Key Numeric Columns Stats
numeric_column_stats = df[["Packet Length", "Anomaly Scores"]].agg(["mean", "median", "std", "min", "max"])
print(numeric_column_stats)

#Correlation Matrix
numeric_cols = ["Packet Length", "Source Port", "Destination Port", "Anomaly Scores"]
corr_matrix = df[numeric_cols].corr()
print(corr_matrix)


        Packet Length  Anomaly Scores
mean       781.452725       50.113473
median     782.000000       50.345000
std        416.044192       28.853598
min         64.000000        0.000000
max       1500.000000      100.000000
                  Packet Length  Source Port  Destination Port  Anomaly Scores
Packet Length          1.000000     0.003657          0.002581       -0.003599
Source Port            0.003657     1.000000         -0.005216        0.004826
Destination Port       0.002581    -0.005216          1.000000       -0.003616
Anomaly Scores        -0.003599     0.004826         -0.003616        1.000000


All numeric columns show near zero correlations, thus we can conclude there are no meaningful linear relationships between packet length, ports, and anomaly scores.